# Smoke test — environment and model

A quick check that everything is set up correctly: the model loads, generation
works, the residual-stream cache has the shape the probes expect, and the chat
template produces a sensible answer.

This is not part of the analysis pipeline — run it first on a new machine.

Each check below raises on failure, so a clean run means every check passed.

## Setup

Shared helpers live in [`common.py`](common.py). Select the `interp` conda
environment as this notebook's kernel — it has `transformer_lens`, `torch`,
`scikit-learn` and `matplotlib` installed.

In [ ]:
import os
import sys
from pathlib import Path

# These notebooks use repo-relative paths ("data/...", "plots/..."), exactly as the
# original scripts did when run from the project root. So make that the working
# directory, and put src/ on the import path so `common` is importable.
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".git").exists()), Path.cwd())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

import common
common.ensure_dirs()
print("Working directory:", Path.cwd())

In [ ]:
import torch

## Configuration

The original script took these as command-line flags; set them here instead.

In [ ]:
MODEL_NAME = os.getenv("MODEL_NAME", common.DEFAULT_MODEL_NAME)
DEVICE = "cpu"
DTYPE = None     # None -> float32 on CPU, bfloat16/float16 on GPU
HF_TOKEN = None  # None -> falls back to HUGGINGFACE_TOKEN from .env

print(f"{MODEL_NAME} on {DEVICE}")

## 1. Environment

Load `.env` and authenticate with Hugging Face (only needed to download a model
that isn't cached yet).

In [ ]:
common.init_env(HF_TOKEN)

## 2. Load the model

Note this uses `from_pretrained` rather than `from_pretrained_no_processing` —
the smoke test checks that the ordinary loading path works. The analysis
notebooks load without processing so activations stay in the model's own basis.

In [ ]:
model = common.load_model(MODEL_NAME, device=DEVICE, dtype=DTYPE, no_processing=False)

## 3. Generation

Raw (non-chat-formatted) generation. The output may not be coherent — we only
care that the forward pass and sampling run.

In [ ]:
gen_tokens = model.to_tokens("The capital of France is")
gen_output = model.generate(gen_tokens, max_new_tokens=10)
print("Raw generation result:", model.to_string(gen_output))

## 4. Residual-stream cache

The probes read `blocks.{layer}.hook_resid_post`, so check the cache is
populated and shaped `[batch, seq_len, d_model]`.

In [ ]:
logits, cache = model.run_with_cache(gen_tokens, names_filter=lambda name: "resid_post" in name)

# Keys may be tuples ("resid_post", idx) or strings containing "resid_post".
key = next(
    (
        k
        for k in cache.keys()
        if (isinstance(k, tuple) and k[0] == "resid_post") or (isinstance(k, str) and "resid_post" in k)
    ),
    None,
)
if key is None:
    raise AssertionError(f"No resid_post cache found; available keys: {list(cache.keys())[:5]}")

shape = cache[key].shape
print("Cache sample shape:", shape)

assert len(shape) == 3, f"Cache tensor shape is not 3D [batch, seq_len, d_model]: {shape}"
assert all(isinstance(d, int) for d in shape), f"Cache tensor shape dimension is not an int: {shape}"
print("Cache shape OK.")

## 5. Chat template

The dataset builder formats every prompt through the chat template, so verify it
applies cleanly and that the model actually answers the question.

In [ ]:
messages = [{"role": "user", "content": "What is the capital of France?"}]
formatted_prompt = model.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print("Formatted chat prompt:", formatted_prompt)

chat_tokens = model.to_tokens(formatted_prompt)
chat_output = model.generate(chat_tokens, max_new_tokens=20)
chat_text = model.to_string(chat_output)
print("Chat generation result:", chat_text)

In [ ]:
chat_str = " ".join(map(str, chat_text)) if isinstance(chat_text, (list, tuple)) else str(chat_text)
assert "Paris" in chat_str, f"Expected 'Paris' in chat output, got: {chat_str}"

print("Chat generation result contains 'Paris'.")
print("All checks completed.")